Criar uma tabela de ORDERS com:
Quantidade de itens, vendendores, lojas, mapeamento de Status e valor total da venda (valor total - desconto)

In [0]:
bronze_path = '/Volumes/bikestore/bronze/bronze_volumes'
silver_path = '/Volumes/bikestore/silver/silver_volumes'

In [0]:
bronze_tables = {
    'tmp_bronze_brands' : f'{bronze_path}/brands/',
    'tmp_bronze_categories' : f'{bronze_path}/categories/',
    'tmp_bronze_customers' : f'{bronze_path}/customers/',
    'tmp_bronze_order_items' : f'{bronze_path}/order_items',
    'tmp_bronze_orders' : f'{bronze_path}/orders/',
    'tmp_bronze_products' : f'{bronze_path}/products/',
    'tmp_bronze_staffs' : f'{bronze_path}/staffs/',
    'tmp_bronze_stocks' : f'{bronze_path}/stocks/',
    'tmp_bronze_stores' : f'{bronze_path}/stores/'
}

for view_name, path in bronze_tables.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name))

In [0]:
df_orders_silver = spark.sql("""
WITH orders_items AS (
SELECT
    order_items.order_id,
    order_items.item_id,
    order_items.product_id,
    order_items.quantity,
    order_items.list_price,
    -- Realizando o calculo de desconto com o total de itens comprados e aplicando a formatação de casas decimais
    ROUND((order_items.list_price * order_items.quantity) * (1 - order_items.discount), 2) AS total_sale,
    order_items.discount
FROM tmp_bronze_order_items order_items
)

SELECT 
    orders.order_id,
    orders.customer_id,
    orders.order_status,
    CASE 
        WHEN orders.order_status = 1 THEN 'Pending'
        WHEN orders.order_status = 2 THEN 'Processing'
        WHEN orders.order_status = 3 THEN 'Shipped'
        WHEN orders.order_status = 4 THEN 'Delivered'
        ELSE 'Unknown'
    END status,
    orders.order_date,
    orders.required_date,
    orders.shipped_date,
    orders.store_id,
    stores.store_name,
    stores.state,
    stores.city,
    staffs.first_name,
    staffs.active,
    staffs.email,
    cteOrder_items.product_id,
    cteOrder_items.quantity,
    cteOrder_items.total_sale,
    cteOrder_items.list_price,
    cteOrder_items.discount
FROM tmp_bronze_orders as orders
LEFT JOIN tmp_bronze_stores stores on orders.store_id = stores.store_id
LEFT JOIN tmp_bronze_staffs staffs on orders.staff_id = staffs.staff_id
LEFT JOIN orders_items cteOrder_items on orders.order_id = cteOrder_items.order_id                        
                             """)

df_orders_silver.write.mode('overwrite').format('delta').option('mergeSchema', 'true').save(silver_path + '/orders')

In [0]:
%sql
-- Criando uma tabela física

CREATE TABLE bikestore.silver.silver_orders
USING DELTA
AS
SELECT *
FROM delta.`/Volumes/bikestore/silver/silver_volumes/orders`;